# 04 - Canny et detection de mouvement

**Objectif de l'etape.** Visualiser les contours de la voiture et confirmer les zones mobiles entre deux frames successives.

**Methode.** Canny detecte les contours forts. La difference d'images utilise `absdiff`, puis un seuillage pour produire un masque binaire de mouvement.

**Lien avec le sujet.** Ces resultats valident visuellement la zone suivie avant le calcul principal par Lucas-Kanade.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'data' / 'car' / 'car-11' / 'img'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

image_files = sorted([p for p in DATASET_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
print('Nombre de frames:', len(image_files))

Nombre de frames: 1661


In [2]:
# Bbox manuelle de depart: ajuster si necessaire ou utiliser cv2.selectROI dans l'interface Tkinter.
# Format: x, y, w, h. Cette valeur ne vient pas d'annotations.
initial_bbox = (535, 300, 220, 105)
print('initial_bbox =', initial_bbox)

initial_bbox = (535, 300, 220, 105)


In [3]:
import cv2
from src.preprocessing import preprocess_image_with_method
from src.edge_detection import canny_on_roi
from src.motion_detection import motion_detection_on_roi, detect_motion_between_frames
from src.visualization import draw_edges_overlay, draw_motion_mask_overlay, save_frame, save_comparison_grid

frame1 = cv2.imread(str(image_files[0]))
frame2 = cv2.imread(str(image_files[1]))
gray1 = preprocess_image_with_method(frame1, method='stretching')['enhanced']
gray2 = preprocess_image_with_method(frame2, method='stretching')['enhanced']

edges = canny_on_roi(gray1, initial_bbox)
motion_roi = motion_detection_on_roi(gray1, gray2, initial_bbox, threshold=25)
motion_full = detect_motion_between_frames(gray1, gray2, threshold=25)

edge_overlay = draw_edges_overlay(frame1, edges, bbox=initial_bbox)
motion_overlay = draw_motion_mask_overlay(frame2, motion_roi['motion_mask'], bbox=initial_bbox)

save_frame(edge_overlay, RESULTS_DIR / 'edge_detection' / 'notebook_canny_overlay.png')
save_frame(motion_full['difference'], RESULTS_DIR / 'motion_detection' / 'notebook_difference.png')
save_frame(motion_full['motion_mask'], RESULTS_DIR / 'motion_detection' / 'notebook_motion_mask.png')
save_frame(motion_overlay, RESULTS_DIR / 'motion_detection' / 'notebook_motion_overlay.png')
save_comparison_grid({'canny overlay': edge_overlay, 'difference': motion_full['difference'], 'motion mask': motion_full['motion_mask'], 'motion overlay': motion_overlay}, RESULTS_DIR / 'final_visualization' / 'notebook_edge_motion_detection.png')

True

**Resultats generes.** Les contours Canny sont sauvegardes dans `results/edge_detection/`; la difference d'images et le motion mask dans `results/motion_detection/`.

**Interpretation.** Canny aide a verifier si les limites detectees correspondent a la voiture, mais il peut contenir des contours parasites. La difference d'images montre les pixels qui changent, souvent autour de la voiture, mais elle reste sensible a l'eclairage et aux variations locales d'intensite.